In [0]:
%pip install xgboost lightgbm -q

In [0]:
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import logging
from sklearn.metrics import (precision_score, recall_score, f1_score,
    average_precision_score, brier_score_loss, roc_auc_score)
from scipy.stats import ks_2samp

logging.getLogger("mlflow.tracking.context.registry").disabled = True

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/src")
from databricks_fraud_model.utils import load_config

cfg = load_config()()

FEATURE_STORE = cfg["feature_store"]["table"]  # fraud_detection_dev.gold.fraud_features

df_all = spark.read.table(FEATURE_STORE).toPandas()

# Gold table tem TransactionDT como TIMESTAMP (convertido via from_unixtime).
# Reverter para INT (segundos desde epoch) para calculos temporais do walk-forward.
if not pd.api.types.is_integer_dtype(df_all["TransactionDT"]):
    df_all["TransactionDT"] = (pd.to_datetime(df_all["TransactionDT"]).astype("int64") // 10**9).astype(int)

train = df_all[df_all["split"] == "train"].drop("split", axis=1).reset_index(drop=True)
val   = df_all[df_all["split"] == "val"].drop("split", axis=1).reset_index(drop=True)
teste = df_all[df_all["split"] == "test"].drop("split", axis=1).reset_index(drop=True)

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

exclude_cols = {"TransactionID", "TransactionDT", "target"}
feature_names = [c for c in train.columns if c not in exclude_cols]
cat_cols = [c for c in feature_names if train[c].dtype == "object"]
num_cols = [c for c in feature_names if c not in cat_cols]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), cat_cols),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
])

preprocessor.fit(train[feature_names])

def get_X(df):
    return preprocessor.transform(df[feature_names])

In [0]:
display(train[["TransactionID", "target"] + feature_names[:5]].head())

In [0]:
def compute_metrics(predictions_pd):
    """Calcula precision, recall, F1, PR-AUC, KS e Brier score.
    Aceita coluna 'proba' (float ja extraido) ou 'probability' (vector)."""
    y_true = predictions_pd["target"].values

    if "proba" in predictions_pd.columns:
        y_proba = predictions_pd["proba"].values.astype(float)
    else:
        y_proba = predictions_pd["probability"].values
        y_proba = np.array([float(p[1]) for p in y_proba])

    y_pred = (y_proba >= 0.5).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    pr_auc = average_precision_score(y_true, y_proba)
    roc_auc = roc_auc_score(y_true, y_proba)
    brier = brier_score_loss(y_true, y_proba)

    # KS estatístico
    fraud_proba = y_proba[y_true == 1]
    legit_proba = y_proba[y_true == 0]
    if len(fraud_proba) > 0 and len(legit_proba) > 0:
        ks_stat, _ = ks_2samp(fraud_proba, legit_proba)
    else:
        ks_stat = 0.0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "pr_auc": round(pr_auc, 4),
        "roc_auc": round(roc_auc, 4),
        "ks_statistic": round(ks_stat, 4),
        "brier_score": round(brier, 4)
    }

In [0]:
df_train = train.copy()
df_train["day_idx"] = (df_train["TransactionDT"] / 86400).astype(int)

In [0]:
min_day = int(df_train["day_idx"].min())
max_day = int(df_train["day_idx"].max())
total_days = max_day - min_day + 1

initial_train_days = int(total_days * 0.6)
step_days = max(1, int(total_days * 0.15))

windows = []
train_start = min_day
train_end = min_day + initial_train_days - 1

while train_end < max_day:
    test_start = train_end + 1
    test_end = min(test_start + step_days - 1, max_day)
    windows.append({
        "train_start": train_start,
        "train_end": train_end,
        "test_start": test_start,
        "test_end": test_end
    })
    train_end = test_end

display(pd.DataFrame(windows))

In [0]:
fraud_ratio = (df_train["target"] == 1).sum() / len(df_train)
weight_fraud = 1.0 / fraud_ratio
weight_legit = 1.0 / (1.0 - fraud_ratio)

df_train["class_weight"] = np.where(df_train["target"] == 1, weight_fraud, weight_legit)

In [0]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

models = {
    "XGBoost": XGBClassifier(
        n_estimators=500,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=50,
        gamma=0.001,
        max_bin=1500,
        random_state=42,
        eval_metric="aucpr",
        n_jobs=-1,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=3,
        min_samples_leaf=50,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=500,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.7,
        min_child_weight=50,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
    "LogisticRegression": LogisticRegression(
        max_iter=100,
        C=0.01,
        penalty="elasticnet",
        l1_ratio=0.5,
        solver="saga",
        random_state=42,
        n_jobs=-1,
    ),
}

In [0]:
import mlflow
from sklearn.base import clone

mlflow.set_experiment("/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/experiments/fraud_detection_walkforward")

all_results = []

for fold_idx, window in enumerate(windows):
    train_fold = df_train[(df_train["day_idx"] >= window["train_start"]) & (df_train["day_idx"] <= window["train_end"])]
    test_fold  = df_train[(df_train["day_idx"] >= window["test_start"]) & (df_train["day_idx"] <= window["test_end"])]
    train_count, test_count = len(train_fold), len(test_fold)
    if test_count == 0 or train_count < 100:
        continue

    X_train_fold = get_X(train_fold)
    y_train_fold = train_fold["target"].values
    w_train_fold = train_fold["class_weight"].values
    X_test_fold = get_X(test_fold)
    y_test_fold = test_fold["target"].values

    for model_name, model in models.items():
        with mlflow.start_run(run_name=f"{model_name}_fold{fold_idx + 1}"):
            mlflow.log_params({"fold": fold_idx + 1, "model": model_name, "train_size": train_count, "test_size": test_count, "cv_type": "walk_forward"})
            fitted_model = clone(model)
            fitted_model.fit(X_train_fold, y_train_fold, sample_weight=w_train_fold)
            proba = fitted_model.predict_proba(X_test_fold)
            pred_pd = pd.DataFrame({"target": y_test_fold, "proba": proba[:, 1]})
            metrics = compute_metrics(pred_pd)
            metrics.update({"fold": fold_idx + 1, "model": model_name, "train_size": train_count, "test_size": test_count})
            all_results.append(metrics)
            mlflow.log_metrics({k: metrics[k] for k in ["precision", "recall", "f1_score", "pr_auc", "roc_auc", "ks_statistic", "brier_score"]})
            print(f"{model_name}_fold{fold_idx + 1} | PR-AUC: {metrics['pr_auc']} | ROC-AUC: {metrics['roc_auc']} | F1: {metrics['f1_score']} | KS: {metrics['ks_statistic']}")

In [0]:
summary_pd = pd.DataFrame(all_results)

summary_rows = []
for model_name in summary_pd["model"].unique():
    model_folds = summary_pd[summary_pd["model"] == model_name]
    row = {"model": model_name}
    for metric in ["precision", "recall", "f1_score", "pr_auc", "roc_auc", "ks_statistic", "brier_score"]:
        row[f"{metric}_mean"] = round(model_folds[metric].mean(), 4)
        row[f"{metric}_std"] = round(model_folds[metric].std(), 4)
    summary_rows.append(row)

    with mlflow.start_run(run_name=f"{model_name}_aggregate"):
        mlflow.log_params({
            "model_name": model_name,
            "cv_type": "walk_forward",
            "n_folds": int(model_folds.shape[0]),
        })
        for metric in ["precision", "recall", "f1_score", "pr_auc", "roc_auc", "ks_statistic", "brier_score"]:
            mlflow.log_metrics({
                f"{metric}_mean": row[f"{metric}_mean"],
                f"{metric}_std": row[f"{metric}_std"],
            })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.sort_values("model"))

**Análise**

O modelo XGBoost otimizado via Optuna demonstrou desempenho superior em todas as métricas comparado aos modelos baseline. Com PR-AUC médio de **0.4856** no walk-forward cross-validation, o XGBoost superou LightGBM (0.4698), RandomForest (0.3127) e Logistic Regression (0.3450). Esta métrica é crucial em problemas de classes desbalanceadas, pois reflete a capacidade do modelo de identificar fraudes mantendo uma proporção aceitável de acertos.

**Resultados no conjunto de teste:**
* **PR-AUC**: 0.505 — O modelo mantém boa capacidade de ordenar transações fraudulentas
* **ROC-AUC**: 0.891 — Excelente discriminação entre classes
* **KS Statistic**: 0.617 — Forte separação entre distribuições de fraude e legítimo
* **Threshold otimizado**: 0.80 maximiza F1-score (0.493)

**Generalização (Validação → Teste):**
O modelo demonstrou **excelente capacidade de generalização**:
* Precision aumentou de 0.559 para 0.660 (+0.10)
* Recall aumentou de 0.347 para 0.393 (+0.05)
* F1-Score aumentou de 0.428 para 0.493 (+0.06)
* FPR (False Positive Rate) diminuiu de 0.97% para 0.84% (-0.13%)

Este comportamento indica que o modelo está bem calibrado e não sofre de overfitting — na verdade, ele **performa melhor no teste** do que na validação, sugerindo que os padrões de fraude no período de teste foram ainda mais distintos dos legítimos.

**Impacto financeiro (com threshold 0.80):**
* **Detecção**: 39.3% das fraudes capturadas (1.844 de 4.689)
* **Precisão**: 66% das transações sinalizadas são realmente fraudes
* **FP Rate**: Apenas 0.84% de falsos alarmes
* **ROI líquido**: R$ 174.880 (assumindo R$100 por fraude detectada e R$10 por investigação de falso positivo)

Portanto, o modelo atinge o objetivo: **encontra os positivos (fraudes) mantendo proporção aceitável de acertos entre os casos sinalizados**, validando a escolha do PR-AUC como métrica de otimização.

### Avaliando o modelo na validacao e no teste

In [0]:
import mlflow
from sklearn.base import clone

default_model = clone(models["XGBoost"])
default_model.fit(get_X(df_train), df_train["target"].values, sample_weight=df_train["class_weight"].values)

for split_name, split_df in [("train", train), ("val", val), ("test", teste)]:
    with mlflow.start_run(run_name=f"XGBoost_default_{split_name}"):
        mlflow.log_params({"model": "XGBoost", "eval_set": split_name, "split_size": len(split_df)})
        proba = default_model.predict_proba(get_X(split_df))
        pred_pd = pd.DataFrame({"target": split_df["target"].values, "proba": proba[:, 1]})
        metrics = compute_metrics(pred_pd)
        mlflow.log_metrics(metrics)
        print(f"\n--- {split_name.upper()} ---")
        for k, v in metrics.items():
            print(f"  {k}: {v}")

### Otimizacao de hiperparametros

In [0]:
import mlflow

In [0]:
import numpy as np
import pandas as pd
import optuna
import mlflow
import mlflow.xgboost

from xgboost import XGBClassifier
from mlflow.models import infer_signature

RANDOM_STATE = 42
N_TRIALS = 20

EXPERIMENT_PATH = "/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/experiments/fraud_detection_walkforward"
mlflow.set_experiment(EXPERIMENT_PATH)
optuna.logging.set_verbosity(optuna.logging.WARNING)


# Dados
X_train = get_X(df_train)
y_train = df_train["target"].to_numpy()
w_train = df_train["class_weight"].to_numpy()

X_val = get_X(val)
y_val = val["target"].to_numpy()


def sanitize_metrics(metrics):
    """Converte métricas para float e remove valores não finitos."""
    return {
        name: float(value)
        for name, value in metrics.items()
        if np.isfinite(float(value))
    }


with mlflow.start_run(run_name="XGBoost_Optuna_HPO"):

    mlflow.set_tags({
        "model_type": "XGBoost",
        "problem_type": "fraud_detection",
        "validation_strategy": "temporal_holdout",
        "optimization_metric": "pr_auc",
        "class_weighting": "sample_weight",
    })

    mlflow.log_params({
        "n_trials": N_TRIALS,
        "random_state": RANDOM_STATE,
    })

    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=100),
            "max_depth": trial.suggest_int("max_depth", 2, 6),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 100.0, log=True),
            "gamma": trial.suggest_float("gamma", 0.0, 10.0),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 100.0, log=True),
            "max_delta_step": trial.suggest_categorical("max_delta_step", [0, 1, 5, 10]),
            "max_bin": trial.suggest_categorical("max_bin", [128, 256, 512]),
        }

        with mlflow.start_run(run_name=f"trial_{trial.number:03d}", nested=True):
            model = XGBClassifier(
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                early_stopping_rounds=50,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbosity=0,
                **params,
            )

            model.fit(
                X_train,
                y_train,
                sample_weight=w_train,
                eval_set=[(X_val, y_val)],
                verbose=False,
            )

            val_proba = model.predict_proba(X_val)[:, 1]
            pred_val = pd.DataFrame({"target": y_val, "proba": val_proba})
            metrics = sanitize_metrics(compute_metrics(pred_val))
            best_iteration = getattr(model, "best_iteration", None)

            mlflow.log_params(params)
            mlflow.log_metrics(metrics)

            if best_iteration is not None:
                mlflow.log_metric("best_iteration", int(best_iteration))

            trial.set_user_attr("best_iteration", best_iteration)
            return metrics["pr_auc"]

    study = optuna.create_study(
        direction="maximize",
        study_name="xgb_pr_auc_v2",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True),
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        gc_after_trial=True,
        show_progress_bar=True,
    )

    # Melhor configuração
    best_params = study.best_params.copy()
    best_iteration = study.best_trial.user_attrs.get("best_iteration")

    final_n_estimators = (
        int(best_iteration) + 1
        if best_iteration is not None
        else int(best_params["n_estimators"])
    )

    final_params = {
        **best_params,
        "n_estimators": final_n_estimators,
    }

    mlflow.log_params({
        f"best_{name}": value
        for name, value in final_params.items()
    })
    mlflow.log_metric("best_validation_pr_auc", float(study.best_value))

    # Treinamento do melhor modelo
    best_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
        **final_params,
    )

    best_model.fit(X_train, y_train, sample_weight=w_train, verbose=False)

    # Avaliação final
    final_proba = best_model.predict_proba(X_val)[:, 1]
    final_pred = pd.DataFrame({"target": y_val, "proba": final_proba})
    final_metrics = sanitize_metrics(compute_metrics(final_pred))

    mlflow.log_metrics({f"final_val_{name}": value for name, value in final_metrics.items()})

    # Registro do modelo
    input_example = X_val.iloc[:5].copy() if hasattr(X_val, "iloc") else X_val[:5].copy()
    signature = infer_signature(input_example, best_model.predict_proba(input_example)[:, 1])

    mlflow.xgboost.log_model(
        xgb_model=best_model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
    )

    print(f"Best PR-AUC: {study.best_value:.4f}")
    print(f"Best trial: {study.best_trial.number}")
    print(f"Best iteration: {best_iteration}")
    print(f"Final n_estimators: {final_n_estimators}")
    print(f"Best params: {final_params}")
    print(f"Validation metrics: {final_metrics}")

In [0]:
# ──────────────────────────────────────────────────────────────
# AVALIAÇÃO DO MODELO OTIMIZADO (best_params do Optuna)
#
# Métricas completas + gráficos:
#   1. Tabela de métricas (train/val/test)
#   2. Curva ROC
#   3. Curva Precision-Recall
#   4. Gráfico KS (ECDF por classe)
#   5. Calibração (reliability diagram)
#   6. Distribuição das probabilidades por classe
# ──────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
import numpy as np
import pandas as pd

# 1. Treina o modelo otimizado com best_params
best_params = study.best_params
print(f"Best PR-AUC (val): {study.best_value:.4f}")
print(f"Best params: {best_params}\n")

best_model = XGBClassifier(
    random_state=42, eval_metric="aucpr", n_jobs=-1, **best_params
)
best_model.fit(X_train, y_train, sample_weight=w_train)

# 2. Coleta predições de cada split
splits = {}
for name, df in [("train", train), ("val", val), ("test", teste)]:
    proba = best_model.predict_proba(get_X(df))
    splits[name] = pd.DataFrame({"target": df["target"].values, "proba": proba[:, 1]})

# 3. Tabela de métricas
print("=" * 70)
print(f"{'Métrica':<18} {'Train':>10} {'Val':>10} {'Test':>10}")
print("=" * 70)
all_metrics = {}
for name, pred_pd in splits.items():
    all_metrics[name] = compute_metrics(pred_pd)
metric_order = ["precision", "recall", "f1_score", "pr_auc", "roc_auc", "ks_statistic", "brier_score"]
for m in metric_order:
    row = f"{m:<18}"
    for name in ["train", "val", "test"]:
        row += f" {all_metrics[name][m]:>10}"
    print(row)
print("=" * 70)

# Usa o TESTE para os gráficos
test_pd = splits["test"]
y_true = test_pd["target"].values
y_proba = test_pd["proba"].values
fraud_proba = y_proba[y_true == 1]
legit_proba = y_proba[y_true == 0]

# ─── GRÁFICOS ───
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("Avaliação do XGBoost Otimizado — Conjunto de Teste", fontsize=16, fontweight="bold")

# (1) Curva ROC
ax = axes[0, 0]
for name in ["train", "val", "test"]:
    pd_split = splits[name]
    fpr, tpr, _ = roc_curve(pd_split["target"], pd_split["proba"])
    ax.plot(fpr, tpr, label=f"{name} (AUC={all_metrics[name]['roc_auc']:.4f})")
ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curva ROC")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

# (2) Curva Precision-Recall
ax = axes[0, 1]
for name in ["train", "val", "test"]:
    pd_split = splits[name]
    p, r, _ = precision_recall_curve(pd_split["target"], pd_split["proba"])
    ax.plot(r, p, label=f"{name} (AP={all_metrics[name]['pr_auc']:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Curva Precision-Recall")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

# (3) Gráfico KS (ECDF por classe)
ax = axes[0, 2]
fraud_sorted = np.sort(fraud_proba)
legit_sorted = np.sort(legit_proba)
fraud_ecdf = np.arange(1, len(fraud_sorted) + 1) / len(fraud_sorted)
legit_ecdf = np.arange(1, len(legit_sorted) + 1) / len(legit_sorted)
ax.plot(legit_sorted, legit_ecdf, label="Legítimo (classe 0)", color="blue", lw=1.5)
ax.plot(fraud_sorted, fraud_ecdf, label="Fraude (classe 1)", color="red", lw=1.5)
# Marca o ponto de KS
ks_stat = all_metrics["test"]["ks_statistic"]
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.4, label=f"threshold=0.5")
ax.set_xlabel("Probabilidade Predita")
ax.set_ylabel("ECDF (proporção acumulada)")
ax.set_title(f"Gráfico KS (estatística={ks_stat:.4f})")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

# (4) Calibração (reliability diagram)
ax = axes[1, 0]
for name in ["val", "test"]:
    pd_split = splits[name]
    frac_pos, mean_pred = calibration_curve(pd_split["target"], pd_split["proba"], n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=f"{name}")
ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.5, label="Perfeitamente calibrado")
ax.set_xlabel("Probabilidade média predita")
ax.set_ylabel("Fração de fraudes observada")
ax.set_title("Diagrama de Calibração")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)

# (5) Distribuição das probabilidades por classe
ax = axes[1, 1]
bins = np.linspace(0, 1, 50)
ax.hist(legit_proba, bins=bins, alpha=0.6, color="blue", label="Legítimo", density=True)
ax.hist(fraud_proba, bins=bins, alpha=0.6, color="red", label="Fraude", density=True)
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="threshold=0.5")
ax.set_xlabel("Probabilidade Predita")
ax.set_ylabel("Densidade")
ax.set_title("Distribuição das Probabilidades por Classe")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

# (6) KS — distância entre ECDFs
ax = axes[1, 2]
all_proba_sorted = np.sort(y_proba)
# Calcula CDFs no mesmo grid
from scipy.stats import ks_2samp
cdf_legit = np.searchsorted(legit_sorted, all_proba_sorted, side="right") / len(legit_sorted)
cdf_fraud = np.searchsorted(fraud_sorted, all_proba_sorted, side="right") / len(fraud_sorted)
ax.plot(all_proba_sorted, cdf_legit, color="blue", label="CDF Legítimo")
ax.plot(all_proba_sorted, cdf_fraud, color="red", label="CDF Fraude")
ax.fill_between(all_proba_sorted, cdf_legit, cdf_fraud, alpha=0.2, color="green", label=f"KS gap ({ks_stat:.4f})")
ax.set_xlabel("Probabilidade Predita")
ax.set_ylabel("CDF acumulada")
ax.set_title("Distância KS entre CDFs")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Cálculo to threshold maximizacao do f1-score

In [0]:
# ──────────────────────────────────────────────────────────────
# OTIMIZACAO DE THRESHOLD — maximizar F1-score
#
# Para cada threshold calcula:
#   TP, FP, TN, FN, precision, recall, F1, FPR
#   Recall$ (valor recuperado por fraudes detectadas)
#   FP$    (custo de falsos positivos)
#   Net$   (retorno liquido)
#   % fraude detectada, % FP
# ──────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# IMPORTANTE: Otimizar threshold na VALIDACAO, nao no teste!
# Usar teste aqui seria data leakage
y_true = splits["val"]["target"].values
y_proba = splits["val"]["proba"].values
n_total = len(y_true)
n_fraud = int(y_true.sum())
n_legit = n_total - n_fraud

# Assumptions financeiras — ajuste conforme o negocio
AVG_FRAUD_VALUE = 100   # R$ recuperado por fraude detectada (TP)
COST_FP         = 10    # R$ custo por falso positivo (investigacao)

thresholds = np.arange(0.01, 1.00, 0.01)
rows = []

for t in thresholds:
    y_pred = (y_proba >= t).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    fpr       = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    recall_dollar  = tp * AVG_FRAUD_VALUE
    fp_dollar      = fp * COST_FP
    net_dollar     = recall_dollar - fp_dollar

    rows.append({
        "threshold":           round(t, 2),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "precision":          round(precision, 4),
        "recall":             round(recall, 4),
        "f1_score":           round(f1, 4),
        "fpr":                round(fpr, 6),
        "fraud_detected_pct": round(recall * 100, 2),
        "fp_rate_pct":        round(fpr * 100, 4),
        "recall_$":           recall_dollar,
        "fp_$":               fp_dollar,
        "net_$":              net_dollar,
    })

thresholds_df = pd.DataFrame(rows)

# Melhor F1
best_idx = thresholds_df["f1_score"].idxmax()
best = thresholds_df.loc[best_idx]
print("=" * 75)
print(f"Melhor F1 -> threshold = {best['threshold']:.2f}")
print(f"  F1={best['f1_score']:.4f} | Recall={best['recall']:.4f} | Precision={best['precision']:.4f} | FPR={best['fpr']:.6f}")
print(f"  TP={best['TP']} | FP={best['FP']} | FN={best['FN']} | TN={best['TN']}")
print(f"  Fraude detectada: {best['fraud_detected_pct']:.2f}% | FP rate: {best['fp_rate_pct']:.4f}%")
print(f"  Recall R$ = R$ {best['recall_$']:,.0f} | FP R$ = R$ {best['fp_$']:,.0f} | Net R$ = R$ {best['net_$']:,.0f}")
print("=" * 75)

display(thresholds_df)

# ─── Matriz de Confusao (melhor threshold) ───
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred_best = (y_proba >= best['threshold']).astype(int)
cm = confusion_matrix(y_true, y_pred_best)

# Matriz de confusao como DataFrame
cm_df = pd.DataFrame(
    cm,
    index=["Real Legítimo", "Real Fraude"],
    columns=["Pred Legítimo", "Pred Fraude"]
)
print("\nMatriz de Confusão:")
display(cm_df)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Leg\u00edtimo", "Fraude"],
            yticklabels=["Leg\u00edtimo", "Fraude"], ax=ax)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confus\u00e3o \u2014 Threshold {best['threshold']:.2f} (F1={best['f1_score']:.4f})")
plt.tight_layout()
plt.show()

In [0]:
# ──────────────────────────────────────────────────────────────
# AVALIACAO DO THRESHOLD OTIMIZADO NO CONJUNTO DE TESTE
#
# Usa o threshold encontrado na validacao (best['threshold'])
# e avalia no teste para medir generalizacao
# ──────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Dados do TESTE
y_true_test = splits["test"]["target"].values
y_proba_test = splits["test"]["proba"].values
n_total_test = len(y_true_test)
n_fraud_test = int(y_true_test.sum())
n_legit_test = n_total_test - n_fraud_test

# Assumptions financeiras (mesmas da validacao)
AVG_FRAUD_VALUE = 100
COST_FP = 10

# Usa o threshold otimizado da validacao
optimal_threshold = best['threshold']
y_pred_test = (y_proba_test >= optimal_threshold).astype(int)

# Calcula metricas no teste
tp = int(((y_pred_test == 1) & (y_true_test == 1)).sum())
fp = int(((y_pred_test == 1) & (y_true_test == 0)).sum())
tn = int(((y_pred_test == 0) & (y_true_test == 0)).sum())
fn = int(((y_pred_test == 0) & (y_true_test == 1)).sum())

precision_test = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall_test = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_test = 2 * precision_test * recall_test / (precision_test + recall_test) if (precision_test + recall_test) > 0 else 0.0
fpr_test = fp / (fp + tn) if (fp + tn) > 0 else 0.0

recall_dollar_test = tp * AVG_FRAUD_VALUE
fp_dollar_test = fp * COST_FP
net_dollar_test = recall_dollar_test - fp_dollar_test

# ─── Comparativo Val vs Teste ───
print("=" * 80)
print(f"AVALIACAO DO THRESHOLD OTIMIZADO (t={optimal_threshold:.2f}) — VALIDACAO vs TESTE")
print("=" * 80)
print(f"{'Metrica':<20} {'Validacao':>15} {'Teste':>15} {'Delta':>15}")
print("-" * 80)

metrics_comparison = [
    ("Precision", best['precision'], precision_test),
    ("Recall", best['recall'], recall_test),
    ("F1-Score", best['f1_score'], f1_test),
    ("FPR", best['fpr'], fpr_test),
    ("TP", int(best['TP']), tp),
    ("FP", int(best['FP']), fp),
    ("TN", int(best['TN']), tn),
    ("FN", int(best['FN']), fn),
]

for metric, val_value, test_value in metrics_comparison:
    if metric in ["TP", "FP", "TN", "FN"]:
        delta = test_value - val_value
        print(f"{metric:<20} {val_value:>15,.0f} {test_value:>15,.0f} {delta:>+15,.0f}")
    else:
        delta = test_value - val_value
        print(f"{metric:<20} {val_value:>15.4f} {test_value:>15.4f} {delta:>+15.4f}")

print("-" * 80)
print(f"{'Recall R$':<20} {best['recall_$']:>15,} {recall_dollar_test:>15,} {recall_dollar_test - best['recall_$']:>+15,}")
print(f"{'FP R$':<20} {best['fp_$']:>15,} {fp_dollar_test:>15,} {fp_dollar_test - best['fp_$']:>+15,}")
print(f"{'Net R$':<20} {best['net_$']:>15,} {net_dollar_test:>15,} {net_dollar_test - best['net_$']:>+15,}")
print("=" * 80)

# ─── Matriz de Confusao (TESTE) ───
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Heatmap
ax = axes[0]
cm_test = confusion_matrix(y_true_test, y_pred_test)
sns.heatmap(cm_test, annot=True, fmt=",d", cmap="Blues",
            xticklabels=["Legítimo", "Fraude"],
            yticklabels=["Legítimo", "Fraude"], ax=ax)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title(f"Matriz de Confusão — Teste\nThreshold {optimal_threshold:.2f} (F1={f1_test:.4f})")

# (2) Comparativo Val vs Teste (barras)
ax = axes[1]
categories = ["Precision", "Recall", "F1-Score"]
val_scores = [best['precision'], best['recall'], best['f1_score']]
test_scores = [precision_test, recall_test, f1_test]

x = np.arange(len(categories))
width = 0.35

ax.bar(x - width/2, val_scores, width, label="Validação", color="skyblue", edgecolor="black")
ax.bar(x + width/2, test_scores, width, label="Teste", color="salmon", edgecolor="black")

ax.set_ylabel("Score")
ax.set_title(f"Generalização do Threshold {optimal_threshold:.2f}")
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend()
ax.grid(alpha=0.3, axis="y")
ax.set_ylim([0, 1])

# Adiciona valores nas barras
for i, (v, t) in enumerate(zip(val_scores, test_scores)):
    ax.text(i - width/2, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
    ax.text(i + width/2, t + 0.02, f"{t:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

# ─── Armazena resultado para uso posterior ───
test_performance = {
    "threshold": optimal_threshold,
    "precision": precision_test,
    "recall": recall_test,
    "f1_score": f1_test,
    "fpr": fpr_test,
    "TP": tp, "FP": fp, "TN": tn, "FN": fn,
    "recall_$": recall_dollar_test,
    "fp_$": fp_dollar_test,
    "net_$": net_dollar_test,
}

print("\nMetricas de teste armazenadas em 'test_performance'")

In [0]:
# ─── Grafico: F1 vs Threshold + Impacto Financeiro ───
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(thresholds_df["threshold"], thresholds_df["f1_score"], label="F1", color="blue", lw=2)
ax.plot(thresholds_df["threshold"], thresholds_df["recall"], label="Recall", color="red", alpha=0.7)
ax.plot(thresholds_df["threshold"], thresholds_df["precision"], label="Precision", color="green", alpha=0.7)
ax.axvline(x=best['threshold'], color="gray", ls="--", alpha=0.5, label=f"Best={best['threshold']:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("F1 / Recall / Precision vs Threshold")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(thresholds_df["threshold"], thresholds_df["recall_$"], label="Recall R$", color="green", lw=2)
ax.plot(thresholds_df["threshold"], thresholds_df["fp_$"], label="FP R$", color="red", lw=2)
ax.plot(thresholds_df["threshold"], thresholds_df["net_$"], label="Net R$", color="blue", lw=2)
ax.axvline(x=best['threshold'], color="gray", ls="--", alpha=0.5, label=f"Best F1={best['threshold']:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("R$")
ax.set_title("Impacto Financeiro vs Threshold")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# ──────────────────────────────────────────────────────────────
# IMPACTO NO NEGOCIO — visualizacoes para tomada de decisao
#
# 1. Cumulative Gains (fraude capturada por decil de score)
# 2. Lift Chart (concentracao de fraude vs aleatorio)
# 3. Net ROI vs Threshold (retorno liquido por threshold)
# 4. Fraude Detectada vs Perdida (bar chart no best threshold)
# 5. R$ recuperado vs R$ investido por decil
# 6. Curva Precision-Recall com zona de operacao
# ──────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Dados do teste (cell 19/21)
y_true = splits["test"]["target"].values
y_proba = splits["test"]["proba"].values
n_total = len(y_true)
n_fraud = int(y_true.sum())

AVG_FRAUD_VALUE = 100  # R$ por fraude detectada
COST_FP         = 10   # R$ por falso positivo

# ─── DataFrame ordenado por score ───
impact_df = pd.DataFrame({"target": y_true, "proba": y_proba})
impact_df = impact_df.sort_values("proba", ascending=False).reset_index(drop=True)
impact_df["decile"] = pd.qcut(impact_df.index, 10, labels=[f"D{i+1}" for i in range(10)])

# ─── Gains e Lift por decil ───
decile_stats = impact_df.groupby("decile", observed=True).agg(
    n_records=("target", "count"),
    n_fraud=("target", "sum"),
    avg_score=("proba", "mean"),
).reset_index()
decile_stats["cum_fraud"] = decile_stats["n_fraud"].cumsum()
decile_stats["cum_fraud_pct"] = decile_stats["cum_fraud"] / n_fraud * 100
decile_stats["lift"] = (decile_stats["n_fraud"] / decile_stats["n_records"]) / (n_fraud / n_total)
decile_stats["rs_recuperado"] = decile_stats["n_fraud"] * AVG_FRAUD_VALUE
decile_stats["rs_investido"] = (decile_stats["n_records"] - decile_stats["n_fraud"]) * COST_FP
decile_stats["rs_liquido"] = decile_stats["rs_recuperado"] - decile_stats["rs_investido"]

# ─── Figura 2x3 ───
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("Impacto do Modelo no Negocio — Identificacao de Fraude", fontsize=16, fontweight="bold")

# (1) Cumulative Gains Chart
ax = axes[0, 0]
ax.plot(np.arange(10, 110, 10), decile_stats["cum_fraud_pct"], marker="o", lw=2, color="blue", label="Modelo")
ax.plot([0, 100], [0, 100], "--", color="gray", alpha=0.5, label="Aleatorio")
ax.fill_between(np.arange(10, 110, 10), decile_stats["cum_fraud_pct"], np.arange(10, 110, 10), alpha=0.15, color="blue")
ax.set_xlabel("% de transacoes avaliadas (top-score)")
ax.set_ylabel("% de fraudes capturadas")
ax.set_title("Cumulative Gains — Fraude Capturada por Decil")
ax.legend()
ax.grid(alpha=0.3)

# (2) Lift Chart
ax = axes[0, 1]
colors = ["red" if l > 1 else "blue" for l in decile_stats["lift"]]
bars = ax.bar(decile_stats["decile"], decile_stats["lift"], color=colors, alpha=0.7, edgecolor="black", linewidth=0.5)
ax.axhline(y=1, color="gray", ls="--", alpha=0.5, label="Aleatorio (lift=1)")
ax.set_xlabel("Decil (D1 = maior score)")
ax.set_ylabel("Lift")
ax.set_title("Lift Chart — Concentracao de Fraude por Decil")
ax.legend()
ax.grid(alpha=0.3)
for bar, val in zip(bars, decile_stats["lift"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.1f}", ha="center", fontsize=9)

# (3) Net ROI vs Threshold
ax = axes[0, 2]
ax.plot(thresholds_df["threshold"], thresholds_df["net_$"], color="blue", lw=2)
ax.axhline(y=0, color="gray", ls="-", alpha=0.3)
best_net_idx = thresholds_df["net_$"].idxmax()
best_net = thresholds_df.loc[best_net_idx]
ax.axvline(x=best["threshold"], color="green", ls="--", alpha=0.6, label=f"Max F1={best['threshold']:.2f}")
ax.axvline(x=best_net["threshold"], color="red", ls="--", alpha=0.6, label=f"Max Net R$={best_net['threshold']:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Net R$")
ax.set_title("Retorno Liquido vs Threshold")
ax.legend()
ax.grid(alpha=0.3)

# (4) Fraude Detectada vs Perdida (no best threshold)
ax = axes[1, 0]
categories = ["Detectada\n(TP)", "Perdida\n(FN)", "Falso Alarme\n(FP)", "Correto\n(TN)"]
values = [int(best["TP"]), int(best["FN"]), int(best["FP"]), int(best["TN"])]
colors_bar = ["green", "red", "orange", "lightblue"]
bars = ax.bar(categories, values, color=colors_bar, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Numero de transacoes")
ax.set_title(f"Decisao no Threshold {best['threshold']:.2f}")
ax.grid(alpha=0.3, axis="y")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01, f"{val:,}", ha="center", fontsize=10, fontweight="bold")

# (5) R$ recuperado vs R$ investido por decil
ax = axes[1, 1]
x = np.arange(len(decile_stats))
w = 0.35
ax.bar(x - w/2, decile_stats["rs_recuperado"], w, label="R$ Recuperado", color="green", alpha=0.7)
ax.bar(x + w/2, decile_stats["rs_investido"], w, label="R$ Investigacao (FP)", color="red", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(decile_stats["decile"])
ax.set_xlabel("Decil (D1 = maior score)")
ax.set_ylabel("R$")
ax.set_title("R$ Recuperado vs R$ Investido por Decil")
ax.legend()
ax.grid(alpha=0.3, axis="y")

# (6) Curva Precision-Recall com zona de operacao
ax = axes[1, 2]
p_curve, r_curve, thr_curve = precision_recall_curve(y_true, y_proba)
ax.plot(r_curve, p_curve, color="blue", lw=2, label="Curva PR")
for t_marker in [0.3, 0.5, best["threshold"]]:
    mask = thr_curve <= t_marker
    if mask.any():
        idx_t = np.where(mask)[0][-1]
        ax.scatter(r_curve[idx_t], p_curve[idx_t], s=80, zorder=5)
        ax.annotate(f"t={t_marker:.2f}", (r_curve[idx_t], p_curve[idx_t]), textcoords="offset points", xytext=(5, 8), fontsize=9)
ax.set_xlabel("Recall (% fraude detectada)")
ax.set_ylabel("Precision (% acerto)")
ax.set_title("Zona de Operacao — Precision vs Recall")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ─── Resumo executivo ───
print("=" * 70)
print("RESUMO EXECUTIVO — Impacto do Modelo")
print("=" * 70)
print(f"Total de transacoes no teste: {n_total:,}")
print(f"Fraudes reais: {n_fraud:,} ({n_fraud/n_total*100:.2f}%)")
print(f"")
print(f"Top 10% (D1) captura: {decile_stats.loc[0, 'n_fraud']:.0f} fraudes ({decile_stats.loc[0, 'n_fraud']/n_fraud*100:.1f}% do total)")
print(f"Top 30% (D1-D3) captura: {decile_stats['cum_fraud'].iloc[2]:.0f} fraudes ({decile_stats['cum_fraud_pct'].iloc[2]:.1f}% do total)")
print(f"")
print(f"Threshold otimo (F1): {best['threshold']:.2f}")
print(f"  Detectadas: {int(best['TP']):,} | Perdidas: {int(best['FN']):,}")
print(f"  R$ recuperado: R$ {best['recall_$']:,.0f} | R$ custo FP: R$ {best['fp_$']:,.0f} | Net: R$ {best['net_$']:,.0f}")
print(f"")
print(f"Threshold otimo (Net R$): {best_net['threshold']:.2f}")
print(f"  Detectadas: {int(best_net['TP']):,} | Perdidas: {int(best_net['FN']):,}")
print(f"  R$ recuperado: R$ {best_net['recall_$']:,.0f} | R$ custo FP: R$ {best_net['fp_$']:,.0f} | Net: R$ {best_net['net_$']:,.0f}")
print("=" * 70)

display(decile_stats)

In [0]:
# ─── Lift Acumulado ───
import numpy as np
import matplotlib.pyplot as plt

# Calcula lift acumulado: % fraudes capturadas / % populacao avaliada
cum_pct_pop = np.arange(10, 110, 10)  # 10%, 20%, ..., 100%
cum_lift = decile_stats["cum_fraud_pct"].values / cum_pct_pop

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# (1) Lift acumulado vs % populacao
ax = axes[0]
ax.plot(cum_pct_pop, cum_lift, marker="o", lw=2, color="blue", label="Lift Acumulado")
ax.axhline(y=1, color="gray", ls="--", alpha=0.5, label="Aleatorio (lift=1)")
ax.fill_between(cum_pct_pop, cum_lift, 1, where=(cum_lift > 1), alpha=0.15, color="blue")
for x, y in zip(cum_pct_pop, cum_lift):
    ax.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
ax.set_xlabel("% de transacoes avaliadas (top-score)")
ax.set_ylabel("Lift Acumulado")
ax.set_title("Lift Acumulado — Concentracao de Fraude")
ax.legend()
ax.grid(alpha=0.3)

# (2) Comparativo: Modelo vs Aleatorio (fraude capturada)
ax = axes[1]
ax.plot(cum_pct_pop, decile_stats["cum_fraud_pct"], marker="o", lw=2, color="blue", label="Modelo")
ax.plot(cum_pct_pop, cum_pct_pop, "--", color="gray", alpha=0.5, label="Aleatorio")
gap = decile_stats["cum_fraud_pct"].values - cum_pct_pop
ax.fill_between(cum_pct_pop, cum_pct_pop, decile_stats["cum_fraud_pct"], alpha=0.15, color="green", label="Ganho")
for x, y, g in zip(cum_pct_pop, decile_stats["cum_fraud_pct"], gap):
    ax.annotate(f"+{g:.1f}%", (x, (x + y) / 2), textcoords="offset points", xytext=(5, 0), fontsize=8, color="green")
ax.set_xlabel("% de transacoes avaliadas")
ax.set_ylabel("% de fraudes capturadas")
ax.set_title("Modelo vs Aleatorio — Fraude Acumulada")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabela
lift_df = pd.DataFrame({
    "% pop avaliada": cum_pct_pop,
    "% fraude capturada": decile_stats["cum_fraud_pct"].round(2),
    "lift acumulado": np.round(cum_lift, 4),
    "ganho vs aleatorio": np.round(gap, 2),
})
print("Lift Acumulado por Decil:")
display(lift_df)

## Perfil Financeiro por Faixa de Risco — Testes Estatísticos

Quando criamos **faixas de risco** (ex: Baixo, Médio, Alto, Crítico) a partir do score do modelo, queremos saber se essa segmentação é **realmente informativa**  ou seja, se a proporção de fraudes muda de forma significativa entre as faixas, ou se qualquer diferença observada poderia ser apenas ruído aleatório.

### Teste Qui-Quadrado de Independência

O **teste qui-quadrado (χ²)** verifica se existe associação entre duas variáveis categóricas — no nosso caso, **faixa de risco** e **target (fraude vs. legítimo)**.

**Hipóteses:**
- **H₀ (nula):** Não há associação — a taxa de fraude é a mesma em todas as faixas de risco
- **H₁ (alternativa):** Existe associação — a taxa de fraude varia entre as faixas

**Como funciona:** O teste compara a frequência **observada** (o que realmente aconteceu) com a frequência **esperada** (o que aconteceria se H₀ fosse verdadeira). Quanto maior a diferença, maior o valor de χ² e menor o p-valor.

### O Problema do Tamanho da Amostra

Com amostras grandes (como as nossas: ~118 mil transações), **quase qualquer diferença mínima se torna estatisticamente significativa** (p-valor < 0,05). Isso acontece porque o valor de χ² cresce proporcionalmente a N (tamanho da amostra). Ou seja, uma diferença de 0,1% na taxa de fraude pode gerar p < 0,0001 simplesmente porque temos muitos dados — **não porque a diferença é prática ou relevante**.

### Cramér's V — O Efeito Que Não Depende de N

Para contornar esse problema, usamos o **Cramér's V**, uma medida de **tamanho de efeito** que é **normalizada pelo tamanho da amostra**:

$$V = \sqrt{\frac{\chi^2}{N \times (k - 1)}}$$

onde *k* é o menor entre o número de linhas e colunas da tabela de contingência.

**Interpretação de V:**

| V | Associação |
|---|---|
| 0,00 – 0,10 | Praticamente nula |
| 0,10 – 0,20 | Fraca |
| 0,20 – 0,30 | Moderada |
| > 0,30 | Forte |

Diferente do p-valor, **Cramér's V não cresce com N** — ele mede a **força** da associação, não apenas se ela existe. Assim, mesmo com 118 mil registros, se V for pequeno, sabemos que a associação é fraca na prática.

### Resumo da Análise

Para cada faixa de risco, calculamos:
- **Taxa de fraude** observada
- **R$ em risco** (fraudes × R$100) e **R$ de investigação** (FP × R$10)
- **Qui-quadrado** (p-valor) — a associação existe?
- **Cramér's V** — a associação é forte?
- **Resíduos padronizados** — qual faixa contribui mais para a associação?

In [0]:
# ──────────────────────────────────────────────────────────────
# PERFIL FINANCEIRO POR FAIXA DE RISCO + TESTES ESTATISTICOS
# ──────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, norm

# Dados do teste
y_true = splits["test"]["target"].values
y_proba = splits["test"]["proba"].values
n_total = len(y_true)
n_fraud = int(y_true.sum())
overall_rate = n_fraud / n_total

AVG_FRAUD_VALUE = 100  # R$ por fraude
COST_FP         = 10   # R$ por falso positivo

# ─── 1. Criar faixas de risco (quintis do score) ───
band_labels = ["Muito Baixo", "Baixo", "Medio", "Alto", "Critico"]
band_cutoffs = np.quantile(y_proba, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
risk_band = pd.cut(y_proba, bins=band_cutoffs, labels=band_labels, include_lowest=True)

risk_df = pd.DataFrame({"target": y_true, "proba": y_proba, "risk_band": risk_band})

# ─── 2. Estatisticas por faixa ───
band_stats = risk_df.groupby("risk_band", observed=True).agg(
    n=("target", "count"),
    fraud=("target", "sum"),
    avg_score=("proba", "mean"),
).reset_index()
band_stats["legit"] = band_stats["n"] - band_stats["fraud"]
band_stats["fraud_rate_pct"] = band_stats["fraud"] / band_stats["n"] * 100
band_stats["rs_risco"] = band_stats["fraud"] * AVG_FRAUD_VALUE
band_stats["rs_investigacao"] = band_stats["legit"] * COST_FP
band_stats["rs_liquido"] = band_stats["rs_risco"] - band_stats["rs_investigacao"]
band_stats["lift"] = band_stats["fraud_rate_pct"] / (overall_rate * 100)

print("=" * 85)
print("PERFIL FINANCEIRO POR FAIXA DE RISCO (conjunto de teste)")
print("=" * 85)
display(band_stats)

# ─── 3. Tabela de contingencia + Qui-Quadrado ───
contingency = pd.crosstab(risk_df["risk_band"], risk_df["target"])
chi2, p_value, dof, expected = chi2_contingency(contingency)

# ─── 4. Cramer's V (tamanho de efeito) ───
k = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n_total * k))

# Interpretacao
if cramers_v < 0.10:
    v_interp = "praticamente nula"
elif cramers_v < 0.20:
    v_interp = "fraca"
elif cramers_v < 0.30:
    v_interp = "moderada"
else:
    v_interp = "forte"

print("\n" + "=" * 85)
print("TESTE QUI-QUADRADO DE INDEPENDENCIA")
print("=" * 85)
print(f"  Chi-quadrado: Chi2 = {chi2:.2f}  |  gl = {dof}  |  p-valor = {p_value:.2e}")
print(f"  Cramer's V:  V = {cramers_v:.4f}  ->  associacao {v_interp}")
print(f"  Interbretacao: {'H0 rejeitada' if p_value < 0.05 else 'H0 nao rejeitada'} (alpha=0.05),")
print(f"  mas o tamanho de efeito ({v_interp}) indica que a relevancia pratica {'e baixa' if cramers_v < 0.20 else 'e relevante'}.")
print("=" * 85)

# ─── 5. Residuos padronizados ───
residuals = (contingency.values - expected) / np.sqrt(expected)
residuals_df = pd.DataFrame(residuals, index=contingency.index, columns=["Legitimo", "Fraude"])
print("\nResiduos Padronizados (>2 = contribui significativamente para a associacao):")
display(residuals_df)

# ─── 6. IC 95% para Cramer's V (bootstrap) ───
np.random.seed(42)
n_boot = 500
v_boot = []
rb_vals = risk_df["risk_band"].values
tg_vals = risk_df["target"].values
for _ in range(n_boot):
    idx = np.random.choice(n_total, n_total, replace=True)
    ct_b = pd.crosstab(rb_vals[idx], tg_vals[idx])
    chi2_b, _, _, _ = chi2_contingency(ct_b)
    v_boot.append(np.sqrt(chi2_b / (n_total * k)))
v_ci = (np.percentile(v_boot, 2.5), np.percentile(v_boot, 97.5))
print(f"\nCramer's V — IC 95% bootstrap: [{v_ci[0]:.4f}, {v_ci[1]:.4f}]")

# ─── 7. Teste de proporcao por faixa (z-test vs taxa geral) ───
print("\n" + "=" * 85)
print(f"TESTE DE PROPORCAO POR FAIXA (vs taxa geral {overall_rate*100:.2f}%):")
print("=" * 85)
print(f"{'Faixa':>12} | {'Taxa':>7} | {'z':>8} | {'p-valor':>10} | {'Sig':>4}")
print("-" * 85)
for _, row in band_stats.iterrows():
    p_band = row["fraud"] / row["n"]
    se = np.sqrt(overall_rate * (1 - overall_rate) / row["n"])
    z = (p_band - overall_rate) / se
    p_z = 2 * (1 - norm.cdf(abs(z)))
    sig = "***" if p_z < 0.001 else "**" if p_z < 0.01 else "*" if p_z < 0.05 else "ns"
    print(f"{row['risk_band']:>12} | {p_band*100:6.2f}% | {z:8.2f} | {p_z:10.2e} | {sig:>4}")
print("*** p<0.001  ** p<0.01  * p<0.05  ns = nao significativo")

# ─── 8. Sensibilidade de H0 ao tamanho da amostra ───
subsample_sizes = [500, 1000, 5000, 10000, 30000, n_total]
sens_results = []
for size in subsample_sizes:
    idx = np.random.choice(n_total, min(size, n_total), replace=False)
    ct_s = pd.crosstab(rb_vals[idx], tg_vals[idx])
    chi2_s, p_s, _, _ = chi2_contingency(ct_s)
    v_s = np.sqrt(chi2_s / (min(size, n_total) * k))
    sens_results.append({"N": size, "chi2": chi2_s, "p_value": p_s, "cramers_v": v_s})
sens_df = pd.DataFrame(sens_results)
print("\n" + "=" * 85)
print("SENSIBILIDADE DE H0 AO TAMANHO DA AMOSTRA:")
print("=" * 85)
print(f"{'N':>8} | {'Chi2':>10} | {'p-valor':>12} | {'Cramer V':>10} | {'Sig?':>6}")
print("-" * 85)
for _, row in sens_df.iterrows():
    sig = "Sim" if row["p_value"] < 0.05 else "Nao"
    print(f"{row['N']:>8,} | {row['chi2']:10.2f} | {row['p_value']:12.2e} | {row['cramers_v']:10.4f} | {sig:>6}")
print("\nObservacao: p-valor cai para 0 conforme N cresce, mas Cramer's V permanece estavel.")
print("Isso demonstra que o p-valor e sensivel a N, mas o tamanho de efeito nao.")
print("=" * 85)

# ─── 9. Visualizacoes ───
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("Perfil Financeiro por Faixa de Risco — Analise Estatistica", fontsize=16, fontweight="bold")
colors = ["green", "yellowgreen", "gold", "orange", "red"]

# (1) Taxa de fraude por faixa
ax = axes[0, 0]
bars = ax.bar(band_stats["risk_band"], band_stats["fraud_rate_pct"], color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=overall_rate*100, color="gray", ls="--", alpha=0.5, label=f"Taxa geral: {overall_rate*100:.2f}%")
ax.set_xlabel("Faixa de Risco")
ax.set_ylabel("Taxa de Fraude (%)")
ax.set_title("Taxa de Fraude por Faixa")
ax.legend()
ax.grid(alpha=0.3, axis="y")
for bar, val in zip(bars, band_stats["fraud_rate_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.2f}%", ha="center", fontsize=10, fontweight="bold")

# (2) R$ por faixa
ax = axes[0, 1]
x = np.arange(len(band_stats))
w = 0.35
ax.bar(x - w/2, band_stats["rs_risco"], w, label="R$ Risco (fraude)", color="red", alpha=0.7)
ax.bar(x + w/2, band_stats["rs_investigacao"], w, label="R$ Investigacao (FP)", color="orange", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(band_stats["risk_band"])
ax.set_xlabel("Faixa de Risco")
ax.set_ylabel("R$")
ax.set_title("Impacto Financeiro por Faixa")
ax.legend()
ax.grid(alpha=0.3, axis="y")

# (3) Residuos padronizados (heatmap)
ax = axes[0, 2]
sns.heatmap(residuals_df, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "Residuo"}, linewidths=0.5)
ax.set_xlabel("Target")
ax.set_ylabel("Faixa de Risco")
ax.set_title(f"Residuos Padronizados\n(Chi2={chi2:.1f}, V={cramers_v:.4f})")

# (4) Lift por faixa
ax = axes[1, 0]
bars = ax.bar(band_stats["risk_band"], band_stats["lift"], color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=1, color="gray", ls="--", alpha=0.5, label="Lift=1 (aleatorio)")
ax.set_xlabel("Faixa de Risco")
ax.set_ylabel("Lift")
ax.set_title("Lift por Faixa de Risco")
ax.legend()
ax.grid(alpha=0.3, axis="y")
for bar, val in zip(bars, band_stats["lift"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f"{val:.1f}x", ha="center", fontsize=10, fontweight="bold")

# (5) Sensibilidade de H0 ao N
ax = axes[1, 1]
ax2 = ax.twinx()
ax.plot(sens_df["N"], sens_df["p_value"], "o-", color="blue", label="p-valor (log)")
ax2.plot(sens_df["N"], sens_df["cramers_v"], "s-", color="red", label="Cramer's V")
ax.set_xlabel("Tamanho da Amostra (N)")
ax.set_ylabel("p-valor", color="blue")
ax2.set_ylabel("Cramer's V", color="red")
ax.set_yscale("log")
ax.set_title("Sensibilidade de H0 ao Tamanho da Amostra")
ax.axhline(y=0.05, color="blue", ls="--", alpha=0.3, label="alpha=0.05")
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc="center right")
ax.grid(alpha=0.3)

# (6) Distribuicao do score por faixa
ax = axes[1, 2]
for i, (label, color) in enumerate(zip(band_labels, colors)):
    mask = risk_df["risk_band"] == label
    ax.hist(y_proba[mask], bins=30, alpha=0.5, color=color, label=label, density=True)
ax.set_xlabel("Score do Modelo")
ax.set_ylabel("Densidade")
ax.set_title("Distribuicao do Score por Faixa")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Análise — Segmentação por Faixa de Risco**

O modelo não distribui risco de forma linear — ele cria uma **linha divisória clara** que concentra fraudes na faixa Crítico (top 20% dos scores).

### 1. Segregação Extrema (Gráficos 1, 4, 6)

**Taxa de Fraude por Faixa** (gráfico 1): crescimento exponencial de 0,20% (Muito Baixo) para **15,65%** (Crítico) — um salto de **78×**. A taxa geral é 3,95%, então apenas a faixa Crítico está acima da média.

**Lift por Faixa** (gráfico 4): as 4 primeiras faixas têm lift **abaixo de 1** (sub-aleatório: 0.1×, 0.1×, 0.3×, 0.6×), enquanto **Crítico atinge 4.0×** — modelo concentra 79% das fraudes em 20% da população.

**Distribuição do Score** (gráfico 6): separação visual clara — faixas Muito Baixo/Baixo agrupam-se perto de 0, Crítico concentra-se acima de 0,6. Essa separação é o que permite a segregação efetiva.

---

### 2. Impacto Financeiro Assimétrico (Gráficos 2, 3)

**Impacto Financeiro** (gráfico 2):
* **Custo de investigação** (FP, laranja): quase constante em ~R\$230k por faixa (cada faixa tem 20% da população)
* **Risco de fraude** (vermelho): microscópico nas 4 primeiras faixas (<R\$55k), mas explode em Crítico (R\$371k)
* **ROI líquido**: **negativo** em todas as faixas exceto Crítico (**R\$171k positivo**)

**Resíduos Padronizados** (gráfico 3, heatmap): Crítico tem resíduo de **+90,59** para Fraude — contribui esmagadoramente para o χ² de 10.835. As 4 primeiras faixas têm resíduos negativos (azul) — **menos fraudes do que esperado** se a distribuição fosse uniforme.

**Implicação operacional**: Investigar manualmente faixas Baixo/Médio/Alto **não compensa** — custo supera benefício. Apenas Crítico justifica intervenção humana.

---

### 3. Validação Estatística (Gráfico 5)

**Sensibilidade de H0 ao N**: O gráfico demonstra o paradoxo da grande amostra:
* **p-valor** (azul, log): cai para ~0 conforme N cresce — com 118k registros, qualquer diferença mínima se torna "significativa"
* **Cramér's V** (vermelho): permanece **estável em ~0.30** (associação **forte**), independente de N

**Conclusão**: A associação entre faixa e fraude é **real e relevante** (V=0.30, forte), não artefato estatístico de amostra grande. O p-valor sozinho seria enganoso.

---

### Resumo Executivo

| **Métrica** | **80% inferior** (Muito Baixo → Alto) | **20% superior** (Crítico) |
|---|---|---|
| **Taxa de fraude** | 0,20% – 2,26% (abaixo da média 3,95%) | **15,65%** (4× acima) |
| **Lift** | 0,1× – 0,6× (sub-aleatório) | **4,0×** |
| **ROI líquido** | Negativo (-R\$178k a -R\$232k) | **+R\$171k** |
| **Resíduo χ²** | Negativos (-13 a -29) | **+90,59** |

**Recomendações:**
1. **Investigação manual**: apenas faixa Crítico (score > 0,6) — único segmento com ROI positivo
2. **Automação**: faixas inferiores podem ter aprovação automática ou regras simples (sem humano)
3. **Threshold**: maximizar cobertura dentro de Crítico, não tentar "pescar" fraudes em faixas inferiores

O modelo cria uma **barreira cirúrgica** que permite alocar recursos de investigação de forma eficiente: ao investigar apenas o top 20% (Crítico), você captura **79%** das fraudes com custo de investigação de apenas 20% da população.

## Interpretabilidade do Modelo

Agora que entendemos o desempenho e o impacto do modelo, vamos investigar como ele toma decisoes analisando a importancia das features e explicando predicoes individuais.

In [0]:
# Feature Importance + Surrogate Model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

print("="*80)
print("PARTE 1: FEATURE IMPORTANCE (XGBoost sklearn)")
print("="*80)

# Extrai importances diretamente do modelo sklearn
fi_array = best_model.feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": fi_array
}).sort_values("importance", ascending=False)

print(f"Total: {len(feature_names)} features")
print(f"\nTop 15:\n{importance_df.head(15).to_string(index=False)}")

print("\n" + "="*80)
print("PARTE 2: SURROGATE SKLEARN XGB")
print("="*80)

# Dados ja em numpy (preprocessados com sklearn)
X_train = get_X(df_train)
y_train = df_train["target"].values
weights_train = df_train["class_weight"].values

X_test = get_X(teste)
y_test = teste["target"].values

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# Treina surrogate
surrogate_model = GradientBoostingClassifier(
    n_estimators=int(best_params["n_estimators"]),
    max_depth=int(best_params["max_depth"]),
    learning_rate=best_params["learning_rate"],
    subsample=best_params["subsample"],
    random_state=42
)

print("Treinando surrogate...")
surrogate_model.fit(X_train, y_train, sample_weight=weights_train)

y_pred_proba = surrogate_model.predict_proba(X_test)[:, 1]
print(f"\nSurrogate metricas (teste):")
print(f"  ROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"  PR AUC:  {average_precision_score(y_test, y_pred_proba):.4f}")
print(f"  F1:      {f1_score(y_test, (y_pred_proba>=0.5).astype(int)):.4f}")
print("\nPronto!")

In [0]:
# SHAP Analysis

import shap
import matplotlib.pyplot as plt
import numpy as np

print("="*80)
print("SHAP ANALYSIS")
print("="*80)

# Usa amostra do teste para performance
np.random.seed(42)
n_sample = min(2000, len(X_test))
sample_idx = np.random.choice(len(X_test), n_sample, replace=False)
X_shap = X_test[sample_idx]
y_shap = y_test[sample_idx]

print(f"Calculando SHAP para {n_sample} transacoes...")
explainer = shap.TreeExplainer(surrogate_model)
shap_values = explainer.shap_values(X_shap)

# Para binaria, pega apenas classe 1 (fraude)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

print(f"SHAP values: {shap_values.shape}")

# Base value
base_val = explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value

# ─── Grafico 1: Summary beeswarm (figura propria) ───
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                  show=False, max_display=15)
plt.title("SHAP Summary Plot (Top 15 Features)", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

# ─── Grafico 2: Bar (mean abs SHAP) ───
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                  plot_type="bar", show=False, max_display=15)
plt.title("SHAP Feature Importance (mean |SHAP value|)", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

# ─── Seleciona exemplos para waterfall ───
fraud_mask = y_shap == 1
fraud_scores = surrogate_model.predict_proba(X_shap[fraud_mask])[:, 1]
high_fraud_idx = np.where(fraud_mask)[0][np.argmax(fraud_scores)]

legit_mask = y_shap == 0
legit_scores = surrogate_model.predict_proba(X_shap[legit_mask])[:, 1]
high_legit_idx = np.where(legit_mask)[0][np.argmax(legit_scores)]

# ─── Grafico 3: Waterfall (fraude com score alto) ───
plt.figure(figsize=(10, 8))
exp_fraud = shap.Explanation(
    values=shap_values[high_fraud_idx],
    base_values=base_val,
    data=X_shap[high_fraud_idx],
    feature_names=feature_names
)
shap.waterfall_plot(exp_fraud, max_display=12, show=False)
plt.title(f"Waterfall: Fraude (score={fraud_scores.max():.3f})", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

# ─── Grafico 4: Waterfall (legitimo com score alto, falso positivo) ───
plt.figure(figsize=(10, 8))
exp_legit = shap.Explanation(
    values=shap_values[high_legit_idx],
    base_values=base_val,
    data=X_shap[high_legit_idx],
    feature_names=feature_names
)
shap.waterfall_plot(exp_legit, max_display=12, show=False)
plt.title(f"Waterfall: Legitimo (score={legit_scores.max():.3f}, FP)", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

# ─── Grafico 5: Dependence plot (feature mais importante) ───
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_feature_idx = mean_abs_shap.argmax()
top_feature_name = feature_names[top_feature_idx]

plt.figure(figsize=(10, 7))
shap.dependence_plot(top_feature_idx, shap_values, X_shap,
                     feature_names=feature_names, show=False)
plt.title(f"SHAP Dependence: {top_feature_name}", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("RESUMO SHAP:")
print("="*80)
shap_importance_df = pd.DataFrame({
    "feature": feature_names,
    "shap_importance": mean_abs_shap
}).sort_values("shap_importance", ascending=False)
print(f"\nTop 10 features (SHAP):")
for i, row in shap_importance_df.head(10).iterrows():
    print(f"  {row['feature']:<30}: {row['shap_importance']:.6f}")
print(f"\nFeature mais importante: {top_feature_name}")
print("\nSHAP concluido!")

In [0]:
# LIME Explanations

%pip install lime -q

from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt

print("="*80)
print("LIME EXPLANATIONS")
print("="*80)

# Cria explainer LIME
print("Criando LIME explainer...")
lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=["Legitimo", "Fraude"],
    mode="classification",
    random_state=42
)

# Seleciona exemplos
fraud_mask_test = y_test == 1
fraud_scores_test = surrogate_model.predict_proba(X_test[fraud_mask_test])[:, 1]
legit_mask_test = y_test == 0
legit_scores_test = surrogate_model.predict_proba(X_test[legit_mask_test])[:, 1]

high_fraud_idx = np.where(fraud_mask_test)[0][np.argmax(fraud_scores_test)]
high_legit_idx = np.where(legit_mask_test)[0][np.argmax(legit_scores_test)]

examples = [
    (high_fraud_idx, "Fraude (score alto)", fraud_scores_test.max()),
    (high_legit_idx, "Legitimo (score alto, FP)", legit_scores_test.max())
]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax_idx, (idx, label, score) in enumerate(examples):
    print(f"\nExplicando: {label} (score={score:.4f})...")
    
    exp = lime_explainer.explain_instance(
        data_row=X_test[idx],
        predict_fn=surrogate_model.predict_proba,
        num_features=12,
        top_labels=1
    )
    
    # Extrai feature contributions
    feature_contrib = exp.as_list(label=1)
    features = [f[0] for f in feature_contrib]
    weights = [f[1] for f in feature_contrib]
    
    # Plot
    ax = axes[ax_idx]
    colors = ["red" if w > 0 else "blue" for w in weights]
    y_pos = np.arange(len(features))
    
    ax.barh(y_pos, weights, color=colors, alpha=0.7, edgecolor="black")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(features, fontsize=9)
    ax.set_xlabel("Contribuicao para Fraude", fontsize=10)
    ax.set_title(f"{label}\nscore={score:.4f}", fontsize=11, fontweight="bold")
    ax.axvline(x=0, color="black", linestyle="--", alpha=0.3)
    ax.invert_yaxis()
    ax.grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("INTERPRETACAO LIME:")
print("  Barras VERMELHAS: features que empurram PARA fraude")
print("  Barras AZUIS: features que empurram PARA legitimo")
print("="*80)